# Structured Output: Getting JSON from an LLM 🧾

So far our LLM hands back **prose** — paragraphs meant for humans to read. That's great for a chatbot, but useless the moment your *code* needs to actually use the answer.

> If you're building an app, you rarely want a paragraph. You want **data**: a price as a number, ingredients as a list, a rating you can store in a database.

This lesson is about forcing the LLM to reply in a clean, predictable shape — **JSON** — that Python can read directly. This is the skill that turns "a cool demo" into "a real app." Builds on lesson 4 (LLM calls) and the classes lesson (you'll see why in a minute).

## 1. The problem: prose is useless to code

Let's ask for a recipe the old way and then try to *use* one specific fact from it — say, the calories.

In [8]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

In [9]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Give me a simple Indian dal recipe (in less than 10 lines) with its calorie count."
)
print(response.text)

Here's a simple, comforting Masoor Dal recipe:

**Simple Masoor Dal**

1.  Wash 1/2 cup masoor dal (red lentils) thoroughly.
2.  Pressure cook with 1.5 cups water, 1/2 tsp turmeric powder, and salt to taste until soft (approx. 3-4 whistles or 15 mins in a pot).
3.  Whisk gently until smooth.
4.  **For Tempering (Tadka):**
5.  Heat 1 tsp ghee or oil in a small pan.
6.  Add 1/2 tsp cumin seeds and a pinch of asafoetida (hing). Let them splutter.
7.  Pour this tempering directly over the cooked dal.
8.  Garnish with fresh coriander leaves (optional). Serve hot!

**Calorie Count (per serving, approx.): 350-400 kcal**


Now imagine your app needs *just the calorie number* to show on a dashboard. How do you pull it out of that paragraph? You can't reliably — it might say "around 250 calories", "250-300 kcal", or bury it in sentence three.

Trying to extract a number from free-form text is like asking a friend for directions and getting a 10-minute story about their cousin's wedding. 🙃 Somewhere in there is the turn you needed... probably.

## 2. First attempt: just *ask* for JSON

**JSON** is simply text in a strict key–value format that every programming language understands. Let's ask the model to use it — and watch a classic gotcha bite us.

In [10]:
import json

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents='''Give me a simple Indian dal recipe.
    Return ONLY JSON with these keys: title, calories, prep_time_minutes.'''
)
print(response.text)   # 👀 look closely at how it starts and ends...

```json
{
  "title": "Simple Red Lentil Dal",
  "calories": 280,
  "prep_time_minutes": 25
}
```


In [11]:
# Looks like JSON, so let's parse it:
data = json.loads(response.text)   # 💥 JSONDecodeError!
print(data)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

💥 **Boom — `JSONDecodeError`.** Why? The model wrapped its answer in a markdown *code fence*:

```
```json
{ "title": "Simple Yellow Dal", "calories": 280, "prep_time_minutes": 10 }
```
```

Those ```` ``` ```` lines are not valid JSON, so `json.loads()` gags on them. The model was *trying* to be helpful (it loves pretty formatting) — but our code doesn't want pretty, it wants parseable.

In [12]:
# Hacky fix: strip the ```json fences off before parsing
clean = response.text.strip()
clean = clean.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

data = json.loads(clean)
print("Calories:", data["calories"])   # finally usable!
print(type(data))

Calories: 280
<class 'dict'>


It works now... but look what we had to do: **babysit the model's output** and scrub the fences by hand. And that's the *easy* failure. Other days it prepends *"Sure, here you go!"*, or adds a helpful note *after* the JSON — and your `removeprefix` hack won't catch those. You'd be playing whack-a-mole forever. 🔨🐹

For a demo, fine. For a real app, we want a **guarantee** — not hope.

## 3. The robust way: tell Gemini the *exact* shape you want

Instead of begging in the prompt, we hand Gemini a **schema** — a precise description of the fields and their types. The model is then *forced* to fill in exactly that shape. No fences, no chit-chat, no surprises.

We describe the shape with a **Pydantic model** — the same data-model class you met in the **classes** lesson. Quick recap: list the fields with their types, and Pydantic enforces them.

In [13]:
from pydantic import BaseModel
from google.genai import types

# This class IS the shape we want back. Note the types: str, int, list of str.
class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    calories: int
    prep_time_minutes: int

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Give me a simple Indian dal recipe.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",   # "reply in JSON"
        response_schema=Recipe,                   # "...in exactly THIS shape"
    ),
)

print(response.text)   # guaranteed-clean JSON string

{"title":"Simple Yellow Dal (Toor Dal)","ingredients":["1 cup toor dal (split pigeon peas)","3 cups water","1/2 tsp turmeric powder","Salt to taste","1 tbsp ghee","1 tsp cumin seeds","1/4 tsp asafoetida (hing)","2 cloves garlic, finely chopped","1-2 dried red chilies (optional)","Fresh cilantro, chopped, for garnish"],"calories":380,"prep_time_minutes":25}


And the best part — because we gave it a schema, the library can hand us back a ready-made `Recipe` **object** (no `json.loads` needed). It's real Python data now:

In [15]:
recipe = response.parsed     # a Recipe object, parsed for us

print("Title:", recipe.title)
print("Calories:", recipe.calories)        # a real int
print("Prep time:", recipe.prep_time_minutes, "min")

print("Ingredients:")
for item in recipe.ingredients:            # a real list we can loop over!
    print(" -", item)

Title: Simple Yellow Dal (Toor Dal)
Calories: 380
Prep time: 25 min
Ingredients:
 - 1 cup toor dal (split pigeon peas)
 - 3 cups water
 - 1/2 tsp turmeric powder
 - Salt to taste
 - 1 tbsp ghee
 - 1 tsp cumin seeds
 - 1/4 tsp asafoetida (hing)
 - 2 cloves garlic, finely chopped
 - 1-2 dried red chilies (optional)
 - Fresh cilantro, chopped, for garnish


🎉 That's the whole game. The LLM's answer is now structured data you can drop into a database, a Streamlit table, an API response, or a calculation. No guessing, no fragile text-parsing.

**Rule of thumb:** chatting with a human → plain text. Feeding the result into code → structured output with a schema.

## 🏋️ Exercise: Customer review analyzer

Companies drown in free-text reviews. Let's turn one into structured data your code can sort and chart.

Given this review:
```
"I ordered the wireless earbuds last week. Sound quality is amazing and battery
lasts all day, but the case feels cheap and it arrived two days late. Probably
won't buy from this brand again."
```

1. Create a Pydantic model `ReviewAnalysis` with these fields:
   - `sentiment`: `str`  (should come back as "positive", "negative", or "neutral")
   - `rating`: `int`  (the model's guess, 1 to 5)
   - `pros`: `list[str]`
   - `cons`: `list[str]`
2. Send the review with `response_schema=ReviewAnalysis`.
3. Print the sentiment and rating, then loop over `pros` and `cons` and print each one.

**Bonus 1:** make a list of 3 different reviews, loop over them, and print a one-line summary for each (`sentiment` + `rating`). Now you've got the core of a review-dashboard. 📊

**Bonus 2:** add a field `recommended_action: str` and watch the model suggest what the company should do.

Hint: the structure is identical to the `Recipe` example — only the fields change. 😉

In [ ]:
# Your solution here
